# Sistem Deteksi Intrusi Jaringan (IDS) Menggunakan Machine Learning
## Klasifikasi Serangan Siber pada Dataset CICIDS 2017

---
**Dataset:** CICIDS 2017 (Canadian Institute for Cybersecurity)  
**Model:** Random Forest + XGBoost + ANN + Ensemble  
**Platform:** Google Colab / Jupyter Notebook


## 1. Install & Import Library

In [ ]:
# Install library yang dibutuhkan
!pip install tensorflow scikit-learn pandas numpy matplotlib seaborn joblib xgboost -q

In [ ]:
# ===== Import Library =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
import urllib.request
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, roc_auc_score, roc_curve
)
import xgboost as xgb

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
np.random.seed(42)
tf.random.set_seed(42)

print(f'TensorFlow: {tf.__version__}')
print('All imports OK!')

## 2. Download Dataset CICIDS 2017

In [ ]:
# ===== Load Dataset CICIDS 2017 (sudah didownload) =====
import os

# Cek file
if os.path.exists('cicids2017_cleaned.csv'):
    print('File cicids2017_cleaned.csv ditemukan!')
    size = os.path.getsize('cicids2017_cleaned.csv') / 1024 / 1024
    print(f'Ukuran: {size:.1f} MB')
elif os.path.exists('/content/cicids2017_cleaned.csv'):
    # Jika di Colab
    import shutil
    shutil.copy('/content/cicids2017_cleaned.csv', '.')
    print('File disalin dari /content/')
else:
    print('File tidak ditemukan!')
    print('Download manual dari Kaggle:')
    print('  kaggle datasets download -d ericanacletoribeiro/cicids2017-cleaned-and-preprocessed --unzip')

## 3. Load & Gabungkan Dataset CICIDS 2017

In [ ]:
# ===== Load Dataset CICIDS 2017 =====
import pandas as pd
import numpy as np

print('Loading cicids2017_cleaned.csv...')
df = pd.read_csv('cicids2017_cleaned.csv', low_memory=False)
df.columns = df.columns.str.strip()

print(f'Shape: {df.shape}')
print(f'Kolom terakhir (label): {df.columns[-1]}')

LABEL_COL = df.columns[-1]  # 'Attack Type'
print(f'\nDistribusi label:')
print(df[LABEL_COL].value_counts())

In [ ]:
# ===== SUBSAMPLE — percepat training (hapus cell ini jika RAM cukup) =====
SAMPLE_SIZE = 300_000

if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f'Subsample aktif: {df.shape}')
else:
    print(f'Dataset sudah kecil, tidak perlu subsample: {df.shape}')

print(f'\nDistribusi setelah subsample:')
print(df[LABEL_COL].value_counts())

In [ ]:
# ===== Info Dataset =====
print(f'Shape: {df.shape}')
print(f'\nJenis traffic:')
vc = df[LABEL_COL].value_counts()
for label, count in vc.items():
    print(f'  {label}: {count:,} ({count/len(df)*100:.1f}%)')
print(f'\nMissing values: {df.isnull().sum().sum():,}')
print(f'Inf values    : {np.isinf(df.select_dtypes(include=np.number)).sum().sum():,}')
df.head(3)

In [ ]:
# ===== Statistik Deskriptif =====
print('Tipe data:')
print(df.dtypes.value_counts())
df.describe().T.head(10)

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# ===== EDA: Distribusi Label =====
import matplotlib.pyplot as plt
import seaborn as sns

# Label di dataset ini: 'Normal Traffic' = benign, sisanya = attack
df['binary_label'] = df[LABEL_COL].apply(
    lambda x: 0 if str(x).strip().lower() in ['normal traffic', 'benign', 'normal'] else 1
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Distribusi Data CICIDS 2017 (Cleaned)', fontsize=15, fontweight='bold')

counts = df['binary_label'].value_counts()
axes[0].pie(counts, labels=['Normal', 'Attack'],
            autopct='%1.1f%%', colors=['#3498DB', '#E74C3C'],
            startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Distribusi Binary Label', fontsize=13, fontweight='bold')

top_labels = df[LABEL_COL].value_counts().head(12)
colors = ['#2ECC71' if str(x).strip().lower() in ['normal traffic','benign','normal'] else '#E74C3C' for x in top_labels.index]
axes[1].barh(top_labels.index[::-1], top_labels.values[::-1], color=colors[::-1])
axes[1].set_title('Distribusi Jenis Traffic', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Jumlah Sampel')
axes[1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('eda_distribusi.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Normal: {counts.get(0,0):,} | Attack: {counts.get(1,0):,}')

In [ ]:
# ===== EDA: Missing & Inf + Top Fitur =====
numeric_df = df.select_dtypes(include=[np.number])
inf_count = np.isinf(numeric_df).sum().sum()
nan_count = df.isnull().sum().sum()
print(f'Inf values: {inf_count:,}')
print(f'NaN values: {nan_count:,}')

# Korelasi top 15 fitur dengan binary label
clean_df = numeric_df.replace([np.inf, -np.inf], np.nan).dropna(axis=1)
clean_df['binary_label'] = df['binary_label']
corr = clean_df.corr()['binary_label'].abs().sort_values(ascending=False)
corr_top = corr.drop('binary_label').head(15)

fig, ax = plt.subplots(figsize=(10, 6))
corr_top[::-1].plot(kind='barh', ax=ax, color='#2563EB', alpha=0.85)
ax.set_title('Top 15 Fitur — Korelasi dengan Label', fontsize=13, fontweight='bold')
ax.set_xlabel('Absolute Correlation')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== EDA: Sample Data =====
print(f'Sample data (5 baris pertama):')
df[[LABEL_COL, 'binary_label']].value_counts().head(10)

In [ ]:
# ===== EDA: Heatmap Korelasi =====
top_feats = corr_top.head(10).index.tolist()
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(clean_df[top_feats].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', ax=ax, linewidths=0.5)
ax.set_title('Heatmap Korelasi Top 10 Fitur CICIDS 2017', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Preprocessing

In [ ]:
# ===== Preprocessing CICIDS 2017 =====
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

print('Preprocessing...')
df_proc = df.copy()

# 1. Binary label
df_proc['binary_label'] = df_proc[LABEL_COL].apply(
    lambda x: 0 if str(x).strip().lower() in ['normal traffic', 'benign', 'normal'] else 1
)

# 2. Replace inf → NaN lalu drop
df_proc.replace([np.inf, -np.inf], np.nan, inplace=True)
before = len(df_proc)
df_proc.dropna(inplace=True)
print(f'  Drop inf/nan: -{before - len(df_proc):,} baris')

# 3. Drop duplikat
before = len(df_proc)
df_proc.drop_duplicates(inplace=True)
print(f'  Drop duplikat: -{before - len(df_proc):,} baris')

# 4. Fitur & label
drop_cols = [LABEL_COL, 'binary_label']
X_df = df_proc.select_dtypes(include=[np.number]).drop(
    [c for c in drop_cols if c in df_proc.select_dtypes(include=[np.number]).columns], axis=1, errors='ignore'
)
feature_columns = X_df.columns.tolist()
X = X_df.values
y = df_proc['binary_label'].values

print(f'\nShape: {X.shape}')
print(f'Fitur: {X.shape[1]}')
print(f'Normal: {sum(y==0):,} | Attack: {sum(y==1):,}')

# 5. Train-Test Split 80/20
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 6. Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled  = scaler.transform(X_test_raw)

joblib.dump(scaler, 'scaler.pkl')
joblib.dump(feature_columns, 'feature_columns.pkl')

print(f'\nX_train: {X_train_scaled.shape}')
print(f'X_test : {X_test_scaled.shape}')
print('Preprocessing selesai!')

## 6. Training Model — Random Forest

In [ ]:
# ===== Random Forest =====
import gc
print('Training Random Forest...')

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf  = rf_model.predict_proba(X_test_scaled)[:, 1]
joblib.dump(rf_model, 'rf_model.pkl')
print(f'RF Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}')
gc.collect()

## 7. Training Model — ANN (Deep Learning)

In [ ]:
# ===== ANN (LIGHTWEIGHT untuk CPU/RAM terbatas) =====
print('Training ANN...')
n_features = X_train_scaled.shape[1]

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = {0: cw[0], 1: cw[1]}
print(f'  Class weights: {class_weights}')

ann_model = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.1),
    Dense(1, activation='sigmoid')
])

ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann_model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
reduce_lr  = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6)

print('\nTraining ANN...')
history = ann_model.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=2048,   # Batch besar → lebih hemat RAM
    validation_split=0.1,
    class_weight=class_weights,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

y_prob_ann = ann_model.predict(X_test_scaled, verbose=0, batch_size=4096).flatten()
y_pred_ann = (y_prob_ann >= 0.5).astype(int)

ann_model.save('ann_model.h5')
print(f'\nANN selesai! Accuracy: {accuracy_score(y_test, y_pred_ann):.4f}')

In [ ]:
# ===== Plot Training History ANN =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Training History ANN', fontsize=15, fontweight='bold')

epochs_ran = range(1, len(history.history['accuracy']) + 1)

# Accuracy
axes[0].plot(epochs_ran, history.history['accuracy'],     'b-o', label='Train Accuracy', linewidth=2, markersize=5)
axes[0].plot(epochs_ran, history.history['val_accuracy'], 'r-s', label='Val Accuracy',   linewidth=2, markersize=5)
axes[0].set_title('Accuracy per Epoch', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Accuracy', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(epochs_ran, history.history['loss'],     'b-o', label='Train Loss', linewidth=2, markersize=5)
axes[1].plot(epochs_ran, history.history['val_loss'], 'r-s', label='Val Loss',   linewidth=2, markersize=5)
axes[1].set_title('Loss per Epoch', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('Loss', fontsize=11)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('ann_training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Evaluasi & Perbandingan Model

In [ ]:
# ===== XGBoost + Model Pembanding (tanpa KNN) =====
import gc
print('Training XGBoost...')
n_neg, n_pos = sum(y_train==0), sum(y_train==1)

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=n_neg/n_pos,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1,
    tree_method='hist'
)
xgb_model.fit(X_train_scaled, y_train)
print('  XGBoost selesai'); gc.collect()

dt_model = DecisionTreeClassifier(max_depth=20, class_weight='balanced', random_state=42)
nb_model = GaussianNB()

dt_model.fit(X_train_scaled, y_train); print('  DT selesai'); gc.collect()
nb_model.fit(X_train_scaled, y_train); print('  NB selesai'); gc.collect()

y_pred_xgb = xgb_model.predict(X_test_scaled)
y_pred_dt  = dt_model.predict(X_test_scaled)
y_pred_nb  = nb_model.predict(X_test_scaled)

y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_prob_dt  = dt_model.predict_proba(X_test_scaled)[:, 1]
y_prob_nb  = nb_model.predict_proba(X_test_scaled)[:, 1]

joblib.dump(xgb_model, 'xgb_model.pkl')
print('Semua model selesai!')

In [ ]:
# ===== Tabel Perbandingan (tanpa KNN) =====
def evaluate_model(y_true, y_pred, y_prob, name):
    return {
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_true, y_pred), 4),
        'Precision': round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_true, y_pred, zero_division=0), 4),
        'F1-Score' : round(f1_score(y_true, y_pred, zero_division=0), 4),
        'ROC-AUC'  : round(roc_auc_score(y_true, y_prob), 4)
    }

# Ensemble soft voting
y_prob_ens = (y_prob_rf + y_prob_xgb + y_prob_ann) / 3
y_pred_ens = (y_prob_ens >= 0.45).astype(int)

results = [
    evaluate_model(y_test, y_pred_rf,  y_prob_rf,  'Random Forest'),
    evaluate_model(y_test, y_pred_ann, y_prob_ann, 'ANN'),
    evaluate_model(y_test, y_pred_xgb, y_prob_xgb, 'XGBoost'),
    evaluate_model(y_test, y_pred_ens, y_prob_ens, 'Ensemble (RF+XGB+ANN)'),
    evaluate_model(y_test, y_pred_dt,  y_prob_dt,  'Decision Tree'),
    evaluate_model(y_test, y_pred_nb,  y_prob_nb,  'Naive Bayes'),
]

df_results = pd.DataFrame(results).set_index('Model')
print('=' * 70)
print('TABEL PERBANDINGAN — CICIDS 2017')
print('=' * 70)
print(df_results.to_string())
print('=' * 70)
best = df_results['Accuracy'].idxmax()
print(f'\nMODEL TERBAIK: {best} ({df_results.loc[best,"Accuracy"]*100:.2f}%)')
df_results

In [ ]:
# ===== Confusion Matrix Side-by-Side (RF dan ANN) =====
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrix — Random Forest vs ANN', fontsize=15, fontweight='bold')

for ax, (y_pred, title, cmap) in zip(axes, [
    (y_pred_rf,  'Random Forest', 'Blues'),
    (y_pred_ann, 'ANN',           'Oranges')
]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Normal', 'Attack'],
                yticklabels=['Normal', 'Attack'],
                linewidths=0.5, cbar=False,
                annot_kws={'size': 14, 'weight': 'bold'})
    ax.set_title(f'{title}\nAccuracy: {accuracy_score(y_test, y_pred):.4f}',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=11)
    ax.set_ylabel('True Label', fontsize=11)

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== ROC-AUC Curve =====
fig, ax = plt.subplots(figsize=(9, 7))

model_data = [
    ('Random Forest',        y_prob_rf,  '#E74C3C'),
    ('ANN',                  y_prob_ann, '#3498DB'),
    ('XGBoost',              y_prob_xgb, '#1ABC9C'),
    ('Ensemble (RF+XGB+ANN)',y_prob_ens, '#E67E22'),
    ('Decision Tree',        y_prob_dt,  '#2ECC71'),
    ('Naive Bayes',          y_prob_nb,  '#F39C12'),
]

for name, y_prob, color in model_data:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, color=color, linewidth=2.5, label=f'{name} (AUC={auc:.4f})')

ax.plot([0,1],[0,1],'k--', linewidth=1.5)
ax.set_title('ROC-AUC Curve — CICIDS 2017', fontsize=14, fontweight='bold')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== Feature Importance (Top 10) =====
feat_df = pd.DataFrame({'Fitur': feature_columns, 'Importance': rf_model.feature_importances_})
feat_df = feat_df.sort_values('Importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, 10))
bars = ax.barh(feat_df['Fitur'][::-1], feat_df['Importance'][::-1], color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, feat_df['Importance'][::-1]):
    ax.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2, f'{val:.4f}', va='center', fontsize=10)
ax.set_title('Top 10 Feature Importance — Random Forest', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ===== Classification Report Detail =====
print('=' * 55)
print('📋 CLASSIFICATION REPORT — RANDOM FOREST')
print('=' * 55)
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Attack']))

print('=' * 55)
print('📋 CLASSIFICATION REPORT — ANN')
print('=' * 55)
print(classification_report(y_test, y_pred_ann, target_names=['Normal', 'Attack']))

## 9. Kesimpulan

In [ ]:
# ===== Ringkasan Akhir — CICIDS 2017 =====
print('=' * 65)
print('RINGKASAN HASIL PENELITIAN — CICIDS 2017')
print('=' * 65)
print(df_results.to_string())
print('=' * 65)
best = df_results['Accuracy'].idxmax()
best_acc = df_results.loc[best, 'Accuracy']
print(f'\nMODEL TERBAIK: {best}')
print(f'Accuracy     : {best_acc:.4f} ({best_acc*100:.2f}%)')
print(f'F1-Score     : {df_results.loc[best, "F1-Score"]:.4f}')
print(f'ROC-AUC      : {df_results.loc[best, "ROC-AUC"]:.4f}')
print(f'''
INFO:
  Dataset : CICIDS 2017 — cicids2017_cleaned.csv (1 file, ~685 MB)
  Split   : 80/20 stratified train_test_split
  Fitur   : {X_train_scaled.shape[1]} fitur numerik
  Label   : 0=Normal Traffic, 1=Attack

FILE DISIMPAN:
  rf_model.pkl        — Random Forest
  ann_model.h5        — ANN (Keras)
  scaler.pkl          — StandardScaler
  feature_columns.pkl — List nama fitur
''')
